#### 结果图1 - ppl

In [ ]:
import matplotlib.pyplot as plt

# 数据
x = [0, 15, 30, 45, 60, 75, 90, 100]
y = [6.413, 6.551, 6.825, 7.03, 7.134, 7.17, 7.189, 7.189]

# 绘制折线图
plt.figure(figsize=(6.4*0.5, 4.8*0.6))
plt.plot(x, y, marker='o', linewidth=2.5, markersize=8, color='steelblue')

# 设置标题和标签
plt.title("WikiText", fontsize=10)
plt.xlabel("Percent of Int4 Experts (%)", fontsize=10)
plt.ylabel("Perplexity", fontsize=10)

# 设置坐标轴刻度与网格
plt.xticks(x)
plt.grid(True, linestyle='--', alpha=0.6)

# 显示数值标注
# for i, val in enumerate(y):
#     plt.text(x[i], val + 0.05, f"{val:.2f}", ha='center', fontsize=10)

# 美化边框
plt.tight_layout()
plt.savefig("wiki_ppl_qwen30b.pdf", format="pdf")


In [ ]:
import matplotlib.pyplot as plt

# 数据
x = [0, 15, 30, 45, 60, 75, 90, 100]
y = [5.74, 5.78, 5.85, 6.01, 6.15, 6.28, 6.36, 6.40]

# 绘制折线图
plt.figure(figsize=(6.4*0.5, 4.8*0.6))
plt.plot(x, y, marker='o', linewidth=2.5, markersize=8, color='steelblue')

# 设置标题和标签
plt.title("WikiText", fontsize=10)
plt.xlabel("Percent of Int2 Experts (%)", fontsize=10)
plt.ylabel("Perplexity", fontsize=10)

# 设置坐标轴刻度与网格
plt.xticks(x)
plt.grid(True, linestyle='--', alpha=0.6)

# 显示数值标注
# for i, val in enumerate(y):
#     plt.text(x[i], val + 0.05, f"{val:.2f}", ha='center', fontsize=10)

# 美化边框
plt.tight_layout()
plt.savefig("wiki_ppl_qwen80b.pdf", format="pdf")


#### 实验结果2 - TTFT and Throughput

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model_data = {
    "Qwen3-30B": {
        "FP16(TP=2)": {"ttft": 0.32, "tpop": 8.46, "prefill": 696.60, "decode": 59.13},
        "Int4(A6000)": {"ttft": 0.28, "tpop": 8.10, "prefill": 1028.91, "decode": 123.56},
        "DynaExq(A6000)": {"ttft": 0.31, "tpop": 9.28, "prefill": 930.28, "decode": 109.33},
        "DynaExq(5090)": {"ttft": 0.16, "tpop": 5.03, "prefill": 1650.19, "decode": 203.24},
    },
    "Qwen3-80B-A3B": {
        "Int4(TP=2)": {"ttft": 0.27, "tpop": 9.15, "prefill": 516.35, "decode": 54.76},
        "Int2(A6000)": {"ttft": 0.30, "tpop": 9.80, "prefill": 900.37, "decode": 100.41},
        "DynaExq(A6000)": {"ttft": 0.33, "tpop": 10.03, "prefill": 845.16, "decode": 91.31},
        "DynaExq(5090)": {"ttft": 0.21, "tpop": 5.87, "prefill": 1535.11, "decode": 176.53},
    },
}

metrics = [
    ("TTFT (s)", "ttft"),
    ("TPOP (ms/token)", "tpop"),
    ("Prefill Throughput (token/s)", "prefill"),
    ("Decode Throughput (token/s)", "decode"),
]

model_colors = {"Qwen3-30B": "#1f77b4", "Qwen3-80B-A3B": "#ff7f0e"}

for title, key in metrics:
    fig, ax = plt.subplots(figsize=(6.4*0.7, 4.8*0.56))
    model_names = list(model_data.keys())
    width = 0.2
    group_spacing = 0.4
    configs_per_model = {name: list(configs.keys()) for name, configs in model_data.items()}

    xtick_positions = []
    xtick_labels = []

    cursor = 0.0
    for idx, model in enumerate(model_names):
        configs = configs_per_model[model]
        positions = cursor + np.arange(len(configs)) * width
        values = [model_data[model][config][key] for config in configs]
        bars = ax.bar(positions, values, width=width * 0.95, color=model_colors[model], alpha=0.9)

        for j, (bar, value, label) in enumerate(zip(bars, values, configs)):
            center = bar.get_x() + bar.get_width() / 2
            # if "Prefill" in title:
            #     ax.text(center, value, f"{value:.2f}", ha="center", va="bottom", fontsize=8)
            # else:
            #     ax.text(center, value, f"{value:.2f}", ha="center", va="bottom", fontsize=10)

            xtick_positions.append(center)
            xtick_labels.append(f"{label}")

        cursor = positions[-1] + width + group_spacing

    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_labels, rotation=20, ha="center", fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    # ax.grid(axis="y", linestyle="--", alpha=0.3)

    handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in model_colors.values()]
    if "TTFT" in title or "TPOP" in title:
        ax.legend(handles, model_colors.keys(), loc="upper left", fontsize=10, bbox_to_anchor=(0.0, 1.03))
    else:
        ax.legend(handles, model_colors.keys(), loc="upper right", fontsize=10, bbox_to_anchor=(1.0, 1.03))

    fig.tight_layout()
    fig.savefig(f"{title.split(' (')[0].replace(' ', '_').replace('/', '_').lower()}_comparison.pdf", format="pdf")
    plt.show()


#### 动态开销